<a href="https://colab.research.google.com/github/PrathamLaddha/Lipophilicity_Prediction_NNs/blob/main/GAT_DMPNN_vs_SOTA_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GAT-DMPNN vs. SOTA Baselines: Same-Split Comparison Pipeline

This notebook sets up a **fair, same-dataset, same-scaffold-split** comparison between:
- Your **GAT-DMPNN** model (results already computed elsewhere)
- **Chemprop (D-MPNN)** — trained/evaluated here on your exact OPERA scaffold split
- **OPERA-style kNN baseline** — reimplemented from scratch here using RDKit descriptors, trained/evaluated on the same split

Run the cells top to bottom. Sections are marked `# EDIT THIS` wherever you need to plug in your own data/paths.

**Roadmap this notebook implements** (see your `comparison_roadmap.tex` doc for full reasoning):
1. Export your exact scaffold split to CSVs
2. Train Chemprop on that split
3. Train an OPERA-style kNN baseline on that split
4. Compute MAE/RMSE/R² + bootstrap CIs for both, using the same metric code
5. Print a final side-by-side comparison table


## 0. Setup: Install dependencies

In [1]:
!pip install -q chemprop rdkit scikit-learn pandas numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.8/151.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.0/305.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 23.2 MB/s eta 0:00:00


## 1. Mount Google Drive (optional)

If your OPERA data / notebook outputs live in Google Drive, mount it here. Skip this cell if you'll upload files directly instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Export your exact scaffold split to CSV

**EDIT THIS SECTION.** This step is the most important one: Chemprop and the OPERA-kNN
baseline must be trained/evaluated on the *exact same molecules* in the *exact same
train/val/test partition* as your GAT-DMPNN model, or the comparison is meaningless.

You need two things from your original notebook:
1. A table of (smiles, logp) for the full deduplicated dataset (after Section
   "Deduplication and Conflict Resolution", before graph construction).
2. The train_idx / val_idx / test_idx arrays your scaffold-split function produced.

If you already saved these to Drive/disk from your original notebook, just point the
paths below at them. Otherwise, re-run your scaffold-split function here (with the
same seed=42) to regenerate the exact same indices.

In [2]:
import pandas as pd
import numpy as np

# --- EDIT THIS: point these at your actual saved files ---
FULL_DATA_PATH = "opera_deduped.csv"      # columns: smiles, logp
TRAIN_IDX_PATH = "train_idx.npy"
VAL_IDX_PATH   = "val_idx.npy"
TEST_IDX_PATH  = "test_idx.npy"
# -----------------------------------------------------------

df = pd.read_csv(FULL_DATA_PATH)
train_idx = np.load(TRAIN_IDX_PATH)
val_idx   = np.load(VAL_IDX_PATH)
test_idx  = np.load(TEST_IDX_PATH)

train_df = df.iloc[train_idx][["smiles", "logp"]].reset_index(drop=True)
val_df   = df.iloc[val_idx][["smiles", "logp"]].reset_index(drop=True)
test_df  = df.iloc[test_idx][["smiles", "logp"]].reset_index(drop=True)

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)

print(f"train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")
train_df.head()


train: 11219 | val: 1402 | test: 1403


,smiles,logp
0,O=C(O)c1c(Cl)cccc1Cl,2.23
1,CC(=O)Oc1ccccc1C(=O)O,1.19
2,O=C(O)c1cc(Cl)ccc1Cl,2.82
3,CC(=O)Nc1ccc(C(=O)O)c(O)c1,1.62
4,CCN(CC)CCNC(=O)c1ccc(N)cc1,0.88


## 3. Train Chemprop (D-MPNN) on the same split

Uses the same batch size / patience / epoch budget as your GAT-DMPNN training protocol
(32 / 40 / 150) so the comparison is fair on training conditions as well as data.

In [3]:
!chemprop train \
  --data-path train.csv val.csv val.csv \
  --task-type regression \
  --smiles-columns smiles \
  --target-columns logp \
  --save-dir chemprop_opera_run \
  --epochs 150 \
  --patience 40 \
  --batch-size 32

Streaming output truncated to the last 5000 lines.
                                                                0.013 val_loss: 
                                                                0.138           
                                                                train_loss_epoc…
Epoch 46/149 ━━━━━━━━━━━╸━━━ 275/351 0:00:02 •       109.53it/s v_num: 0.000    
                                     0:00:01                    train_loss_step:
                                                                0.013 val_loss: 
                                                                0.138           
                                                                train_loss_epoc…
Epoch 46/149 ━━━━━━━━━━━╸━━━ 276/351 0:00:02 •       109.53it/s v_num: 0.000    
                                     0:00:01                    train_loss_step:
                                                                0.014 val_loss: 
                                                          

In [4]:
!chemprop predict \
  --test-path test.csv \
  --model-paths chemprop_opera_run \
  --preds-path chemprop_opera_run/test_preds.csv

chemprop_preds = pd.read_csv("chemprop_opera_run/test_preds.csv")
chemprop_preds.head()


2026-07-18T14:32:49 - INFO:chemprop.cli.main - Running in mode 'predict' with args: {'smiles_columns': None, 'reaction_columns': None, 'no_header_row': False, 'num_workers': 0, 'batch_size': 64, 'accelerator': 'auto', 'devices': 'auto', 'rxn_mode': 'REAC_DIFF', 'multi_hot_atom_featurizer_mode': 'V2', 'keep_h': False, 'add_h': False, 'ignore_stereo': False, 'reorder_atoms': False, 'molecule_featurizers': None, 'descriptors_path': None, 'descriptors_columns': None, 'no_descriptor_scaling': False, 'no_atom_feature_scaling': False, 'no_atom_descriptor_scaling': False, 'no_bond_feature_scaling': False, 'no_bond_descriptor_scaling': False, 'atom_features_path': None, 'atom_descriptors_path': None, 'bond_features_path': None, 'bond_descriptors_path': None, 'constraints_path': None, 'constraints_to_targets': None, 'use_cuikmolmaker_featurization': False, 'test_path': PosixPath('test.csv'), 'output': PosixPath('chemprop_opera_run/test_preds.csv'), 'drop_extra_columns': False, 'model_paths': [Po

,smiles,logp
0,c1ccc(Cn2ccnn2)cc1,0.944704
1,O=c1c2ccccc2nc2n1CCCCC2,1.769300
2,CCCCCCOC(=O)C1=CCC(=O)NC(=O)N1,1.692775
3,COC1C(OC(C)=O)CC(=O)OC(C)CC2OC2/C=C\C(=O)C(C)C...,2.626352
4,c1ccc2nc(N3CCNCC3)ccc2c1,1.318927


## 4. OPERA-style kNN baseline (from-scratch reimplementation)

OPERA's original logP model uses 9 PaDEL descriptors + a distance-weighted kNN
regressor. PaDEL isn't easily scriptable from Python, so this reimplementation
substitutes 9 RDKit descriptors chosen to mirror OPERA's descriptor categories
(size, polarity, branching, aromaticity, electrotopological character). This should
be described in your paper as an RDKit-based reimplementation of OPERA's design
philosophy, not a byte-for-byte reproduction of the original PaDEL-based model.

$k$ is tuned on the validation set, mirroring OPERA's own validation-tuned approach.

In [5]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def compute_descriptors(smiles: str) -> np.ndarray:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.full(9, np.nan)
    return np.array([
        Descriptors.MolWt(mol),
        rdMolDescriptors.CalcNumRotatableBonds(mol),
        rdMolDescriptors.CalcTPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        rdMolDescriptors.CalcFractionCSP3(mol),
        Crippen.MolMR(mol),
        rdMolDescriptors.CalcNumRings(mol),
    ], dtype=float)

def featurize(df, smiles_col="smiles"):
    return np.stack([compute_descriptors(s) for s in df[smiles_col]])

print("Computing RDKit descriptors...")
X_train = featurize(train_df)
X_val   = featurize(val_df)
X_test  = featurize(test_df)

y_train = train_df["logp"].to_numpy()
y_val   = val_df["logp"].to_numpy()
y_test  = test_df["logp"].to_numpy()

def clean(X, y):
    mask = ~np.isnan(X).any(axis=1)
    return X[mask], y[mask]

X_train, y_train = clean(X_train, y_train)
X_val, y_val     = clean(X_val, y_val)
X_test, y_test   = clean(X_test, y_test)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

best_k, best_val_rmse = 5, np.inf
for k in [3, 5, 7, 9, 11, 15]:
    knn = KNeighborsRegressor(n_neighbors=k, weights="distance")
    knn.fit(X_train_s, y_train)
    val_pred = knn.predict(X_val_s)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f"  k={k:2d}  val RMSE={val_rmse:.4f}")
    if val_rmse < best_val_rmse:
        best_val_rmse, best_k = val_rmse, k

print(f"\nBest k: {best_k}")

knn = KNeighborsRegressor(n_neighbors=best_k, weights="distance")
knn.fit(X_train_s, y_train)
opera_knn_pred = knn.predict(X_test_s)


Computing RDKit descriptors...
  k= 3  val RMSE=1.1963
  k= 5  val RMSE=1.1712
  k= 7  val RMSE=1.1612
  k= 9  val RMSE=1.1587
  k=11  val RMSE=1.1574
  k=15  val RMSE=1.1624

Best k: 11


## 5. Compute metrics for both baselines (same code, same definitions)

Includes a 1000-resample bootstrap 95% CI so point estimates aren't over-interpreted
when comparing against your GAT-DMPNN numbers.

In [6]:
def evaluate(y_true, y_pred, name, n_boot=1000, seed=42):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    rng = np.random.default_rng(seed)
    n = len(y_true)
    boot_mae, boot_rmse, boot_r2 = [], [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_mae.append(mean_absolute_error(y_true[idx], y_pred[idx]))
        boot_rmse.append(np.sqrt(mean_squared_error(y_true[idx], y_pred[idx])))
        boot_r2.append(r2_score(y_true[idx], y_pred[idx]))

    ci = lambda v: np.percentile(v, [2.5, 97.5])

    print(f"=== {name} ===")
    print(f"MAE:  {mae:.4f}   95% CI [{ci(boot_mae)[0]:.4f}, {ci(boot_mae)[1]:.4f}]")
    print(f"RMSE: {rmse:.4f}   95% CI [{ci(boot_rmse)[0]:.4f}, {ci(boot_rmse)[1]:.4f}]")
    print(f"R^2:  {r2:.4f}   95% CI [{ci(boot_r2)[0]:.4f}, {ci(boot_r2)[1]:.4f}]")
    print()
    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

results = []

# Chemprop -- adjust the column name below if Chemprop names its output column differently
chemprop_col = [c for c in chemprop_preds.columns if c != "smiles"][0]
results.append(evaluate(test_df["logp"].to_numpy(), chemprop_preds[chemprop_col].to_numpy(), "Chemprop (D-MPNN)"))

# OPERA-style kNN
results.append(evaluate(y_test, opera_knn_pred, "OPERA-style kNN"))

# --- EDIT THIS: paste in your GAT-DMPNN test results for the final comparison table ---
results.append({"model": "GAT-DMPNN (ours)", "MAE": 0.4838, "RMSE": 0.6880, "R2": 0.8530})


=== Chemprop (D-MPNN) ===
MAE:  0.4775   95% CI [0.4537, 0.5048]
RMSE: 0.6658   95% CI [0.6219, 0.7105]
R^2:  0.8624   95% CI [0.8404, 0.8819]

=== OPERA-style kNN ===
MAE:  0.9840   95% CI [0.9367, 1.0280]
RMSE: 1.3000   95% CI [1.2340, 1.3630]
R^2:  0.4753   95% CI [0.4196, 0.5207]



## 6. Final comparison table

In [7]:
comparison_df = pd.DataFrame(results)[["model", "MAE", "RMSE", "R2"]]
comparison_df = comparison_df.sort_values("RMSE").reset_index(drop=True)
comparison_df


,model,MAE,RMSE,R2
0,Chemprop (D-MPNN),0.477529,0.665816,0.862363
1,GAT-DMPNN (ours),0.483800,0.688000,0.853000
2,OPERA-style kNN,0.983959,1.300005,0.475292


## 7. Notes for your paper

- This comparison is now a genuine **Tier A/B (same-dataset, same-split)** comparison for
  Chemprop and the OPERA-style kNN baseline, upgrading them from the literature-only
  "Tier C positioning" caveat used previously.
- State explicitly in your methods section that the OPERA-kNN baseline is an
  **RDKit-based reimplementation** of OPERA's descriptor philosophy, not the original
  PaDEL-based model.
- Uni-Mol is intentionally left out of this notebook (3D conformer generation +
  large pretrained checkpoint fine-tuning is a much heavier lift) — keep citing its
  literature numbers as a Tier C positioning comparison, with the caveat already noted
  in your results doc.